# Homework 7, Q3: Optimizing Through a Physics Simulator with JAX

The first cell defines the simulator (`simulate`) and the plotting helper (`plot_trajectory`). Do not modify it.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

print("JAX version:", jax.__version__)

# DO NOT MODIFY THE CONTENTS OF THIS CELL

# Simulation parameters
num_steps=200 # total number of simulation steps
dt=0.05 # time step size
g=9.8 # gravity acceleration constant
bounce=0.8 # bounce energy retention factor

def step(state):
    """
    One physics step for a bouncing ball.
    Args:
        state: jnp.array([x, y, vx, vy])
    Returns:
        new_state: jnp.array([x_new, y_new, vx_new, vy_new])
    """
    x, y, vx, vy = state

    # integrate
    x_new = x + vx * dt
    y_new = y + vy * dt - 0.5 * g * dt**2
    vx_new = vx
    vy_new = vy - g * dt # gravity effect

    # bounce condition 
    bounced = y_new < 0 # next y position below ground (<0.0)
    y_new = jnp.where(bounced, 0.0, y_new) # reset y position to ground (0.0)
    vy_new = jnp.where(bounced, -bounce * vy_new, vy_new) # reverse and reduce y velocity
    vx_new = jnp.where(bounced, bounce * vx_new, vx_new) # reduce x velocity

    return jnp.array([x_new, y_new, vx_new, vy_new])


def simulate(init_state):
    """
    Simulate the bouncing ball over time.
    Args:
        init_state: jnp.array([x0, y0, vx0, vy0])
    Returns:
        trajectory: jnp.array of shape (num_steps+1, 4)
    """ 
    def body_fn(state, _):
        new_state = step(state)
        return new_state, new_state  # carry + record

    # Run the simulation using jax.lax.scan
    _, trajectory = jax.lax.scan(body_fn, init_state, None, length=num_steps)
    trajectory = jnp.vstack([init_state, trajectory])  # include initial state
    return trajectory  # shape: (num_steps+1, 4)

def plot_trajectory(traj, title="Bouncing Ball Trajectory"):
    """
    Plot (x, y) positions of the simulated ball.
    Args:
        traj: jnp.array of shape (T, 4)
        target: optional (x_target, y_target) tuple for visualization
    """
    x, y = traj[:, 0], traj[:, 1]

    plt.figure(figsize=(6, 4))
    plt.plot(x, y, label="trajectory")
    plt.axhline(0, color="black", lw=1)
    plt.xlabel("x position")
    plt.ylabel("y position")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# Example trajectory

# initial position (0,5) (on y axis) and initial velocity (1,0) (to the right)
init_state = jnp.array([0.0, 5.0, 1.0, 0.0]) 
traj = simulate(init_state)
print(traj.shape)
plot_trajectory(traj)


# Start your solutions below this cell